In [3]:
import os, sys, multiprocessing, pickle
from tqdm import tqdm
import pandas as pd
import numpy as np
from PIL import Image
from scipy.ndimage import binary_dilation, binary_erosion
from skimage.morphology import disk
from matplotlib import pyplot as plt
from scipy import ndimage
from skimage import measure, morphology
import cv2
from scipy import stats
Image.MAX_IMAGE_PIXELS = None

sys.path.append("/DATA/F2FMatcher_DDC")
from config.ddc_config import *
from collections import abc

In [4]:
fs_mask = [
    'area','perimeter','bbox_height', 'bbox_width', 'bbox_aspect_ratio',
    'eccentricity','solidity','extent','major_axis_length','minor_axis_length',
    'orientation','roundness','edge_pixels','edge_density','compactness'
]
fs_intensity = ['mean', 'std', 'p10', 'p25', 'p50', 'p75', 'p90', 'skew', 'kurt']

In [5]:
def get_filename(name_sample, list_names):
    names = [n for n in list_names if name_sample.upper() in n.upper()]
    if len(names)==0:
        print(f'No image of sample {name_sample} found.')
        return None
    else:
        name = names[0].split(".")[0]
        return name
    
# get all images
dict_all_images = {"TA": {}, "QUA": {}}
dict_img2metadata = {"TA": {}, "QUA": {}}

for muscle in ["TA", "QUA"]:
    dir_czi_source = CZI_BASE_DIR_TA if muscle == "TA" else CZI_BASE_DIR_QUA
    dir_CP_MASKS = CP_MASKS_DIR_TA if muscle == "TA" else CP_MASKS_DIR_QUA
    dir_pair_output = PAIR_DIRS_BASE_TA if muscle == "TA" else PAIR_DIRS_BASE_QUA
    list_samples = TA_SAMPLES if muscle == "TA" else QUA_SAMPLES
    
    for sample in list_samples:
        dict_all_images[muscle][sample] = {}
        for slide in SLIDES.keys():
            list_images = [f.split(".czi")[0] for f in os.listdir(dir_czi_source / f'{SLIDES[slide]["czi_dir"]}') \
                            if f.endswith(".czi")]
            img = get_filename(sample, list_images)
            dict_all_images[muscle][sample][slide] = img
            dict_img2metadata[muscle][img] = {"sample": sample, "slide": slide}

list_features = [f"M_{f}" for f in fs_mask]

muscle = 'TA'
sample = "TAG02"
list_samples = TA_SAMPLES if muscle == "TA" else QUA_SAMPLES

for slide in SLIDES.keys():
    dir_save_features = Path(f"/DATA/F2FMatcher_DDC/results/{muscle}/features/slide_{slide}/")
    img = dict_all_images[muscle][sample][slide]
    
    list_channels = SLIDES[slide]["stainings"].keys()

    # concatenate features intensity across channel
    for channel in list_channels:
        for compartement in range(4):
            list_features.extend([f"I_sl{slide}_ch{channel}_c{compartement}_{f}" for f in fs_intensity])

assert len(list_features) == 15+36*18

In [6]:
15+36*18


663